# 👋 Welcome to Atlas JupyterHub

You made it. You're running inside your own container on Atlas — a real research cluster with 2× GV100 GPUs and 256 GB RAM.

## Your environment

- **CPU:** 2 cores (soft cap on interactive container)
- **Memory:** 8 GB (hard cap)
- **GPU:** none in this container by default — use the `gpu` Slurm partition if your Quest needs one
- **Home:** `/home/jovyan/work` is bind-mounted to your Linux home on Atlas — files persist and Slurm jobs see them
- **Idle timeout:** 1 hour — save often

## What's pre-installed

- Python 3.11 with scipy, scikit-learn, pandas, matplotlib
- PyTorch 2.4 (CPU build)
- `erisml-lib` with `[ieip]` extras
- HuggingFace `transformers`, `datasets`
- Slurm client tools (`sinfo`, `squeue`, `scancel`) — but `sbatch` is gated by polite-submit
- `polite-submit` — the submission wrapper you use for every batch job

## Environment smoke check

Run the cell below. If anything fails, paste the output in Discord.

In [ ]:
import subprocess
import sys

print(f'Python: {sys.version.split()[0]}')

try:
    import torch
    print(f'PyTorch: {torch.__version__} (CUDA: {torch.cuda.is_available()})')
except ImportError as e:
    print(f'PyTorch: {e}')

try:
    import erisml.ieip  # noqa: F401
    print('erisml.ieip: OK')
except ImportError as e:
    print(f'erisml.ieip: {e}')

for cmd in ('sinfo', 'squeue', 'polite-submit --version'):
    r = subprocess.run(cmd.split(), capture_output=True, text=True, timeout=10)
    line = (r.stdout + r.stderr).strip().split('\n')[0]
    print(f'{cmd!r}: {"ok" if r.returncode == 0 else "FAIL"} — {line[:80]}')

## 🚀 Submitting jobs — the polite way

Atlas is shared. You, me, Andrew's research, production services — same two GPUs. So direct `sbatch` is disabled. Use `polite-submit` instead:

```bash
polite-submit job.sh
polite-submit --dry-run job.sh            # preview
polite-submit --array sweep.sh --range 0-9 --chunk 3
```

`polite-submit` checks cluster state before submitting. If the queue is busy, it backs off. Your defaults live in `~/polite_submit.yaml` — sensible for Atlas, feel free to tune per project.

### Partitions you can use

| Partition | Use it for | Max time | Max resources |
|---|---|---|---|
| `interactive` | quick test from notebook | 30 min | 2 CPU, 8 GB |
| `cpu` | CPU work | 4 hours | 8 CPU, 32 GB |
| `gpu` | GPU training / inference | 4 hours | 1 GPU, 8 CPU, 64 GB |

The `gpu` partition requires opt-in — ask Andrew on Discord and he'll add you to `gpu_users.txt`.

### Your first job

Create a file called `hello.sh` in this notebook's file browser with this content:

```bash
#!/bin/bash
#SBATCH --job-name=hello
#SBATCH --partition=interactive
#SBATCH --time=00:02:00
#SBATCH --output=hello-%j.out

echo "Hello from $(hostname) at $(date)"
nvidia-smi 2>/dev/null | head -20 || echo '(no GPU on interactive partition)'
```

Then run:

```bash
polite-submit hello.sh
squeue -u $USER          # see your job
cat hello-*.out          # read the output once it completes
```

If anything goes sideways, Discord.

## Where to start

If you're here for Quests:

```bash
git clone https://github.com/ahb-sjsu/agi-hpc.git
cd agi-hpc
```

Open `docs/notebooks/Student_Quests.ipynb` and follow it.

## Housekeeping

- Your home directory persists on Atlas. Don't `rm -rf /` 🙃
- HuggingFace caches land in `~/.cache/huggingface`. You have 8 GB in-container; heavy downloads should use Slurm jobs with more memory.
- If your kernel hangs, restart it from the Kernel menu — your files are safe.
- If the whole container is stuck, close the tab and log in again.

## Questions?

Discord — the same channel where Mr. Wesley invited you.

Good hunting.

*— Andrew*